In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    while current != current.parent:
        if (current / 'src' / 'models' / 'adapter.py').exists():
            return current
        current = current.parent
    raise RuntimeError('Could not locate DriftDetect repo root from the current working directory.')


REPO_ROOT = find_repo_root()
RESULTS_PATH = REPO_ROOT / 'results' / 'tables' / 'decoder_d2_results.npz'
FIGURE_DIR = REPO_ROOT / 'results' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

BAND_LABELS = {
    'dc_trend': 'DC/Trend',
    'very_low': 'Very Low',
    'low': 'Low',
    'mid': 'Mid',
    'high': 'High',
}
BAND_COLORS = {
    'dc_trend': '#7f1d1d',
    'very_low': '#d97706',
    'low': '#1d4ed8',
    'mid': '#15803d',
    'high': '#7c3aed',
}

with np.load(RESULTS_PATH) as data:
    obs_delta = data['obs_delta'].astype(np.float64)
    obs_delta_std = data['obs_delta_std'].astype(np.float64)
    band_names = [str(name) for name in data['band_names'].tolist()]
    epsilons = data['epsilons'].astype(np.float64)
    n_anchors = int(np.asarray(data['n_anchors']))

lipschitz = obs_delta / epsilons[None, :]
eps1_index = int(np.argmin(np.abs(epsilons - 1.0)))

print(f'Loaded {RESULTS_PATH.relative_to(REPO_ROOT)}')
print(f'obs_delta shape: {obs_delta.shape}')
print(f'obs_delta_std shape: {obs_delta_std.shape}')
print(f'band_names: {band_names}')
print(f'epsilons: {epsilons.tolist()}')
print(f'n_anchors: {n_anchors}')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

for band_index, band in enumerate(band_names):
    color = BAND_COLORS.get(band, None)
    mean = obs_delta[band_index]
    std = obs_delta_std[band_index]
    ax.plot(
        epsilons,
        mean,
        marker='o',
        linewidth=2.0,
        color=color,
        label=BAND_LABELS.get(band, band),
    )
    ax.fill_between(
        epsilons,
        np.maximum(mean - std, 0.0),
        mean + std,
        color=color,
        alpha=0.14,
        linewidth=0,
    )

ax.set_xscale('log')
ax.set_xlabel('Perturbation Magnitude (epsilon)')
ax.set_ylabel('Obs L2 Response')
ax.set_title('D2: Decoder Sensitivity by Frequency Band (Lipschitz Probe)')
ax.legend(frameon=True)
ax.grid(True, which='both', alpha=0.25, linewidth=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
sensitivity_fig_path = FIGURE_DIR / 'd2_lipschitz_probe.pdf'
fig.savefig(sensitivity_fig_path, bbox_inches='tight')
plt.show()
print(f'Saved figure: {sensitivity_fig_path.relative_to(REPO_ROOT)}')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

for band_index, band in enumerate(band_names):
    ax.plot(
        epsilons,
        lipschitz[band_index],
        marker='o',
        linewidth=2.0,
        color=BAND_COLORS.get(band, None),
        label=BAND_LABELS.get(band, band),
    )

ax.set_xscale('log')
ax.set_xlabel('Perturbation Magnitude (epsilon)')
ax.set_ylabel('Effective Lipschitz Constant (obs_delta / epsilon)')
ax.set_title('Effective Lipschitz Constant by Frequency Band')
ax.legend(frameon=True)
ax.grid(True, which='both', alpha=0.25, linewidth=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
lipschitz_fig_path = FIGURE_DIR / 'd2_lipschitz_constant.pdf'
fig.savefig(lipschitz_fig_path, bbox_inches='tight')
plt.show()
print(f'Saved figure: {lipschitz_fig_path.relative_to(REPO_ROOT)}')


In [ ]:
eps1 = epsilons[eps1_index]
eps1_delta = obs_delta[:, eps1_index]
eps1_std = obs_delta_std[:, eps1_index]
x = np.arange(len(band_names))
bar_colors = [
    '#dc2626' if band == 'dc_trend' else '#9ca3af'
    for band in band_names
]

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
ax.bar(
    x,
    eps1_delta,
    yerr=eps1_std,
    color=bar_colors,
    edgecolor='#374151',
    linewidth=0.7,
    capsize=4,
)

ax.set_xticks(x)
ax.set_xticklabels([BAND_LABELS.get(band, band) for band in band_names], rotation=20, ha='right')
ax.set_ylabel('Obs L2 Response')
ax.set_title('Obs Response to Unit Perturbation by Band Direction')
ax.grid(True, axis='y', alpha=0.25, linewidth=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

min_index = int(np.argmin(eps1_delta))
ax.text(
    x[min_index],
    eps1_delta[min_index] + eps1_std[min_index] + 0.04,
    'smallest',
    ha='center',
    va='bottom',
    fontsize=9,
    color='#7f1d1d',
)

fig.tight_layout()
bar_fig_path = FIGURE_DIR / 'd2_bar_eps1.pdf'
fig.savefig(bar_fig_path, bbox_inches='tight')
plt.show()
print(f'Saved figure: {bar_fig_path.relative_to(REPO_ROOT)}')


In [ ]:
print('D2 obs_delta: mean obs L2 response')
print()
print(f"{'Band':<12}" + ''.join(f"  eps={epsilon:<8g}" for epsilon in epsilons))
for band_index, band in enumerate(band_names):
    row = f"{band:<12}"
    for epsilon_index in range(len(epsilons)):
        row += f"  {obs_delta[band_index, epsilon_index]:<12.4f}"
    print(row)

print()
print('D2 effective Lipschitz constants: obs_delta / epsilon')
print()
print(f"{'Band':<12}" + ''.join(f"  eps={epsilon:<8g}" for epsilon in epsilons))
for band_index, band in enumerate(band_names):
    row = f"{band:<12}"
    for epsilon_index in range(len(epsilons)):
        row += f"  {lipschitz[band_index, epsilon_index]:<12.4f}"
    print(row)

ranking = sorted(
    zip(band_names, eps1_delta),
    key=lambda item: item[1],
)

print()
print(f'Band ranking by sensitivity at eps={eps1:g} (smallest to largest):')
for rank, (band, value) in enumerate(ranking, start=1):
    print(f'  {rank}. {band:<10} {value:.4f}')


## D2 Result Summary

- `dc_trend` has the **smallest** decoder sensitivity / effective Lipschitz constant among all frequency-band directions.
- This means decoder amplification is **not** directionally biased toward `dc_trend`.
- Combined with D1: the trained decoder amplifies drift overall by roughly 48%, but D2 suggests that amplification is not preferentially aligned with `dc_trend` directions.
- Implication: `dc_trend` dominance in observation space is likely an intrinsic latent property, not a decoder artifact.
- This partially contradicts the original decoder amplification hypothesis (Finding E).
- The result is still valuable because it strengthens the case that trend dominance comes from RSSM latent dynamics, such as saturation or attractor behavior, rather than from the decoder mapping.
- Next step: D3 (linear vs nonlinear separation) will further test the decoder's role.
